# 第7回 フーリエ解析とスペクトル推定

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：系列の中に、どの周期の波がどれだけ入っているかを、どう測るか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "fable", "feasts"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(fable)
library(feasts)
library(ggtime)   # ACFと予測の図（autoplot）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)

### A-1. 波を足し合わせて、系列を作る

これまでは、系列を時間の順に見てきました。今日は、同じ系列を「いろいろな周期の波の足し合わせ」として見ます。

1つの波は、3つの数で決まります。

- 振幅（amplitude）$A$：波の高さ
- 周波数（frequency）$\omega$：1時点のあいだに、波が何回くり返すか。周期（period）は $1/\omega$ 時点
- 位相（phase）$\varphi$：波の始まりの位置

$$
A \cos(2\pi\omega t + \varphi)
$$

2つの波とノイズを足して、長さ200の系列を作ります。

- 波1：周期20（周波数 $1/20 = 0.05$）、振幅3
- 波2：周期8（周波数 $1/8 = 0.125$）、振幅2、位相 $-1$
- ノイズ：標準偏差1のホワイトノイズ

In [ ]:
n <- 200
t <- 1:n
wave1 <- 3 * cos(2 * pi * t / 20)          # 周期20、振幅3
wave2 <- 2 * cos(2 * pi * t / 8 - 1)       # 周期8、振幅2、位相 -1
set.seed(1)
noise <- rnorm(n)                          # ノイズ
x <- wave1 + wave2 + noise                 # 足し合わせた系列
part_names <- c("wave 1 (period 20)", "wave 2 (period 8)", "noise", "x = wave 1 + wave 2 + noise")
parts <- data.frame(t = rep(t, 4),
                    part = factor(rep(part_names, each = n), levels = part_names),
                    value = c(wave1, wave2, noise, x))
ggplot(parts, aes(t, value)) +
  geom_line() +
  facet_wrap(~part, ncol = 1) +
  labs(x = "Time t (step)", y = "Value")

いちばん下の `x` だけを見て、上の2つの波（周期と振幅）を言い当てるのが、今日の課題です。

### A-2. 波の強さを測る：ピリオドグラム

ある周波数 $\omega$ の波が系列にどれだけ入っているかは、系列と、その周波数の $\cos$・$\sin$ との積の平均で測れます。

$$
a = \frac{2}{n} \sum_{t=1}^{n} x_t \cos(2\pi\omega t), \qquad
b = \frac{2}{n} \sum_{t=1}^{n} x_t \sin(2\pi\omega t), \qquad
\text{振幅} = \sqrt{a^2 + b^2}
$$

$\cos$ と $\sin$ の2つを使うのは、位相が分からないからです。$A\cos(2\pi\omega t + \varphi) = a\cos(2\pi\omega t) + b\sin(2\pi\omega t)$ と書き直せます。

In [ ]:
# 周波数 freq の波が、系列 x にどれだけ入っているか
wave_size <- function(x, freq) {
  t <- seq_along(x)
  a <- 2 * mean(x * cos(2 * pi * freq * t))   # cos の係数
  b <- 2 * mean(x * sin(2 * pi * freq * t))   # sin の係数
  c(freq = freq, period = 1 / freq, a = a, b = b, amplitude = sqrt(a^2 + b^2))
}
round(t(sapply(c(1/50, 1/20, 1/10, 1/8, 1/4), function(f) wave_size(x, f))), 3)
# 同じ係数は、cos と sin を説明変数にした回帰でも出る（周期20の場合）
round(coef(lm(x ~ cos(2 * pi * t / 20) + sin(2 * pi * t / 20))), 3)

- 周期20では振幅がほぼ3、周期8ではほぼ2と出ます。作ったときの値です。
- 周期50、10、4では、振幅は0に近くなります。その周期の波は、入っていません。

これを、すべての周波数で一度に行うのが、離散フーリエ変換（discrete Fourier transform, DFT）です。
長さ $n$ の系列では、周波数 $\omega_j = j/n$（$j = 0, 1, \dots, n-1$）を調べます。これをフーリエ周波数（Fourier frequency）と呼びます。

$$
d(\omega_j) = \frac{1}{\sqrt{n}} \sum_{t=1}^{n} x_t\, e^{-2\pi i \omega_j t}, \qquad
I(\omega_j) = \left| d(\omega_j) \right|^2
$$

$I(\omega_j)$ をピリオドグラム（periodogram）と呼びます。周波数 $\omega_j$ の波の強さです。
$e^{-2\pi i \omega t} = \cos(2\pi\omega t) - i \sin(2\pi\omega t)$ なので、$d(\omega_j)$ は上の $a$ と $b$ を1つの複素数にまとめたものです。

Rの `fft()` は、DFTを速く計算します（高速フーリエ変換、fast Fourier transform, FFT）。

In [ ]:
# ピリオドグラム：周波数 j/n（j = 1, 2, ...）ごとの、波の強さ
pgram <- function(x) {
  n <- length(x)
  d <- fft(x - mean(x)) / sqrt(n)       # 離散フーリエ変換（平均を引いてから）
  j <- 1:floor((n - 1) / 2)             # 周波数が 0 より大きく、0.5 より小さいところだけを見る
  data.frame(freq = j / n, period = n / j, spec = Mod(d[j + 1])^2)
}

In [ ]:
pg <- pgram(x)
ggplot(pg, aes(freq, spec)) +
  geom_line() +
  geom_point(size = 0.8) +
  labs(x = "Frequency (cycles per step)", y = "Periodogram")
head(pg[order(-pg$spec), ], 4)       # 高い順に4つ

山は2つだけです。周波数0.05（周期20）と0.125（周期8）にあります。ノイズにまぎれて見えにくかった2つの波が、はっきり分かれました。

山の高さ、系列の分散、Rに付属の関数との関係を確かめます。

In [ ]:
# 山の高さは、n × 振幅^2 / 4
rbind(periodogram = pg$spec[pg$period %in% c(20, 8)], formula = n * c(3, 2)^2 / 4)
# ピリオドグラムを足し合わせると、系列の分散になる
c(variance = mean((x - mean(x))^2), from_periodogram = 2 * sum(pg$spec) / n)
# Rの関数 spec.pgram() と同じ値
sp <- spec.pgram(x, taper = 0, detrend = FALSE, demean = TRUE, fast = FALSE, plot = FALSE)
all.equal(sp$spec[seq_along(pg$spec)], pg$spec)

- 山の高さは、振幅の2乗に比例します。ノイズがあるので、式の値から少しずれます。
- ピリオドグラムは、系列の分散を、周波数ごとに分けたものです（2倍するのは、周波数 $\omega$ と $1 - \omega$ が同じ高さになるからです。周波数0.5の分を数えていないので、わずかに小さく出ます）。

### A-3. 見つけた波を、先へ伸ばす

周期が分かれば、その周期の $\cos$ と $\sin$ を説明変数にした回帰で波を当てはめ、先へ伸ばして予測できます。
最後の40時点をテストデータにし、第2回のベンチマーク（平均法、ナイーブ法）と比べます。

`fourier(period = 20, K = 1)` は、周期20の $\cos$ と $\sin$ の組を説明変数に入れる、という意味です。

In [ ]:
sim <- tsibble(t = t, x = x, index = t)
h_sim <- 40
train_sim <- sim |> filter(t <= n - h_sim)
pg_train <- pgram(train_sim$x)
head(pg_train[order(-pg_train$spec), ], 2)      # 訓練データだけで、周期を見つける
fit_sim <- train_sim |>
  model(mean  = MEAN(x),
        naive = NAIVE(x),
        wave  = TSLM(x ~ fourier(period = 20, K = 1) + fourier(period = 8, K = 1)))
fc_sim <- fit_sim |> forecast(h = h_sim)
autoplot(fc_sim, sim |> filter(t > n - 3 * h_sim), level = NULL) +
  labs(x = "Time t (step)", y = "Value x_t", colour = "Method")
accuracy(fc_sim, sim) |> select(.model, RMSE, MAE)

波を当てはめた予測（`wave`）のRMSEは、ノイズの標準偏差（1）とほぼ同じです。これ以上は当てられません。
波の形が先まで変わらないなら、周期を見つけることが、そのまま予測になります。

### A-4. エイリアシング：速すぎる波は、遅い波に化ける

ピリオドグラムでは、周波数0.5より上を見ませんでした。理由を確かめます。

周波数0.8の波（1時点のあいだに0.8回くり返す）と、周波数0.2の波を、$t = 1, 2, \dots$ で観測します。

In [ ]:
fast <- cos(2 * pi * 0.8 * t)       # 周波数0.8の波を、整数の時点で観測する
slow <- cos(2 * pi * 0.2 * t)       # 周波数0.2の波
max(abs(fast - slow))               # 観測した値の差
pg_fast <- pgram(fast)
pg_fast[which.max(pg_fast$spec), ]  # ピリオドグラムの山の位置

観測した値は、まったく同じです（差は計算の誤差だけ）。ピリオドグラムの山は、0.8ではなく0.2に出ます。

- 観測の間隔が1のとき、見分けられる周波数は0.5までです。これをナイキスト周波数（Nyquist frequency）と呼びます。周期でいえば、2時点より短い周期は見えません。
- それより速い波は、遅い波として現れます。これをエイリアシング（aliasing）と呼びます。
- 周波数 $\omega$ の波と $1 - \omega$ の波は、整数の時点では同じ値になります。

### A-5. スペクトル密度：ACFを、周波数の側から見る

A-1の系列は、決まった波にノイズを足したものでした。第3回のARMAモデルのような確率的な系列には、決まった波はありません。
それでも、「どの周波数の動きが強いか」は決まります。それを表すのが、スペクトル密度（spectral density）です。

スペクトル密度は、自己共分散関数 $\gamma(h)$ のフーリエ変換です。

$$
f(\omega) = \sum_{h=-\infty}^{\infty} \gamma(h)\, e^{-2\pi i \omega h} = \gamma(0) + 2 \sum_{h=1}^{\infty} \gamma(h) \cos(2\pi\omega h)
$$

ARMA(p, q)モデルでは、係数から直接計算できます。

$$
f(\omega) = \sigma_w^2\, \frac{\left| 1 + \theta_1 z + \dots + \theta_q z^q \right|^2}{\left| 1 - \phi_1 z - \dots - \phi_p z^p \right|^2}, \qquad z = e^{-2\pi i \omega}
$$

In [ ]:
# ARMAモデルのスペクトル密度（freq：周波数、ar・ma：係数、sigma2：ノイズの分散）
arma_spec <- function(freq, ar = numeric(0), ma = numeric(0), sigma2 = 1) {
  # 多項式 1 + c_1 z + c_2 z^2 + ... の、z = exp(-2πi f) での値
  poly <- function(coef, f) 1 + sum(coef * exp(-2i * pi * f * seq_along(coef)))
  sigma2 * sapply(freq, function(f) Mod(poly(ma, f))^2 / Mod(poly(-ar, f))^2)
}

4つのモデルで、理論ACFとスペクトル密度を並べます。4つ目は、第3回で使ったAR(2)です。

In [ ]:
models <- list(
  "1 white noise"       = list(ar = numeric(0),   ma = 0),            # x_t = w_t
  "2 AR(1), phi = 0.7"  = list(ar = 0.7,          ma = numeric(0)),   # x_t = 0.7 x_{t-1} + w_t
  "3 AR(1), phi = -0.7" = list(ar = -0.7,         ma = numeric(0)),   # x_t = -0.7 x_{t-1} + w_t
  "4 AR(2)"             = list(ar = c(1.0, -0.5), ma = numeric(0))    # x_t = x_{t-1} - 0.5 x_{t-2} + w_t
)

In [ ]:
acf_four <- do.call(rbind, lapply(names(models), function(m)
  data.frame(model = m, lag = 0:20,
             acf = as.numeric(ARMAacf(models[[m]]$ar, models[[m]]$ma, lag.max = 20)))))
ggplot(acf_four, aes(lag, acf)) +
  geom_hline(yintercept = 0, colour = "grey60") +
  geom_col(width = 0.15) +
  geom_point(size = 0.8) +
  facet_wrap(~model) +
  labs(x = "Lag h (step)", y = "Theoretical ACF")

In [ ]:
freq <- seq(0, 0.5, 0.005)
spec_four <- do.call(rbind, lapply(names(models), function(m)
  data.frame(model = m, freq = freq, spec = arma_spec(freq, models[[m]]$ar, models[[m]]$ma))))
ggplot(spec_four, aes(freq, spec)) +
  geom_line() +
  facet_wrap(~model, scales = "free_y") +
  labs(x = "Frequency (cycles per step)", y = "Spectral density")

- ホワイトノイズ：ACFはラグ0だけ。スペクトル密度は平ら（どの周波数も同じ強さ）です。
- AR(1)、$\phi = 0.7$：ACFはゆっくり小さくなります。スペクトル密度は、低い周波数（ゆっくりした動き）が強くなります。
- AR(1)、$\phi = -0.7$：ACFは1時点ごとに符号が替わります。スペクトル密度は、高い周波数（速い上下）が強くなります。
- AR(2)：ACFは波を打ちます。スペクトル密度は、途中に山があります。

AR(2)で、スペクトル密度が自己共分散のフーリエ変換になっていることを、数値で確かめます。
自己共分散から計算した値（点）と、係数から計算した値（線）を重ねます。

In [ ]:
ar2 <- c(1.0, -0.5)
gamma0 <- sum(c(1, ARMAtoMA(ar2, lag.max = 500))^2)       # 分散 γ(0)（ノイズの分散は1）
gam <- gamma0 * ARMAacf(ar2, lag.max = 50)[-1]             # 自己共分散 γ(1), ..., γ(50)
from_acov <- sapply(freq, function(f) gamma0 + 2 * sum(gam * cos(2 * pi * f * 1:50)))
ggplot(data.frame(freq = freq, from_acov = from_acov, from_coef = arma_spec(freq, ar2)), aes(freq)) +
  geom_line(aes(y = from_coef), colour = "red") +
  geom_point(aes(y = from_acov), size = 0.8) +
  labs(x = "Frequency (cycles per step)", y = "Spectral density (point: ACF, line: coef.)")
max(abs(from_acov - arma_spec(freq, ar2)))                 # 2つの計算の差
fine <- seq(0.001, 0.5, 0.001)
peak <- fine[which.max(arma_spec(fine, ar2))]              # 山の位置
c(peak_freq = peak, peak_period = 1 / peak)

2つは一致します。ACFとスペクトル密度は、同じ情報を、別の向きから見たものです。
山は周波数0.115（周期は約8.7時点）にあります。このAR(2)の系列は、決まった波ではありませんが、9時点前後の周期で上下しやすい、ということです。

### A-6. ピリオドグラムは、ばらつく

データからスペクトル密度を推定します。まず、ピリオドグラムをそのまま使ってみます。
AR(2)から長さ512の系列を作り、ピリオドグラム（灰色）に、理論のスペクトル密度（赤）を重ねます。

In [ ]:
set.seed(7)
n_ar <- 512
x_ar2 <- as.numeric(arima.sim(list(ar = ar2), n = n_ar))
pg_ar2 <- pgram(x_ar2)
pg_ar2$truth <- arma_spec(pg_ar2$freq, ar2)
ggplot(pg_ar2, aes(freq)) +
  geom_line(aes(y = spec), colour = "grey50") +
  geom_line(aes(y = truth), colour = "red", linewidth = 1) +
  labs(x = "Frequency (cycles per step)", y = "Periodogram (red: true spectral density)")

ピリオドグラムは、赤い線のまわりにありますが、大きくばらつきます。

ピリオドグラムは、標本の自己共分散 $\hat{\gamma}(h)$ のフーリエ変換と、ちょうど同じ値です。

In [ ]:
gam_hat <- as.numeric(acf(x_ar2, lag.max = n_ar - 1, type = "covariance", plot = FALSE)$acf)   # γ^(0), ..., γ^(n-1)
from_acov_hat <- sapply(pg_ar2$freq, function(f)
  gam_hat[1] + 2 * sum(gam_hat[-1] * cos(2 * pi * f * 1:(n_ar - 1))))
max(abs(from_acov_hat - pg_ar2$spec))       # 2つの計算の差

系列を長くすれば、ばらつきは減るでしょうか。長さを256、1024、4096と変えます。
予想を決めてから、実行してください。

In [ ]:
set.seed(2)
by_n <- do.call(rbind, lapply(c(256, 1024, 4096), function(n_i) {
  pg_i <- pgram(as.numeric(arima.sim(list(ar = ar2), n = n_i)))
  data.frame(length = n_i, panel = sprintf("n = %d", n_i), pg_i, truth = arma_spec(pg_i$freq, ar2))
}))
by_n$panel <- factor(by_n$panel, levels = unique(by_n$panel))
ggplot(by_n, aes(freq)) +
  geom_line(aes(y = spec), colour = "grey50", linewidth = 0.3) +
  geom_line(aes(y = truth), colour = "red") +
  facet_wrap(~panel) +
  labs(x = "Frequency (cycles per step)", y = "Periodogram (red: true spectral density)")
# ピリオドグラム ÷ 本当のスペクトル密度 の、平均と標準偏差
by_n |>
  group_by(length) |>
  summarise(frequencies = n(), mean_ratio = mean(spec / truth), sd_ratio = sd(spec / truth))

- 系列を長くしても、ばらつきは減りません。比の標準偏差は、どの長さでもほぼ1です。
- 系列を長くすると、調べる周波数の数（`frequencies`）も同じ割合で増えます。1つの周波数に使える情報の量は、増えません。
- 比の平均はほぼ1です。平均的には、正しい高さを指しています。

### A-7. 平滑化：となりの周波数と平均する

ばらつきを減らすために、となり合う $2m + 1$ 個の周波数のピリオドグラムを平均します。これを平滑化（smoothing）と呼びます。
第2回の移動平均を、時間の軸ではなく、周波数の軸で行うことにあたります。

$$
\hat{f}(\omega_j) = \sum_{k=-m}^{m} W_k\, I(\omega_{j+k}), \qquad W_k = \frac{1}{2m+1}
$$

重み $W_k$ を、窓関数（window function）またはスペクトル窓（spectral window）と呼びます。どの周波数にも同じ重みを置くこの窓を、ダニエル窓（Daniell window）と呼びます。

In [ ]:
# 平滑化：となり合う 2m+1 個の周波数の平均に置き換える（両端のm個は、相手がそろわないので除く）
smooth_pgram <- function(pg, m) {
  if (m == 0) return(pg)
  pg$spec <- as.numeric(stats::filter(pg$spec, rep(1 / (2 * m + 1), 2 * m + 1)))
  pg[!is.na(pg$spec), ]
}
# ホワイトノイズの（平滑化した）ピリオドグラム ÷ ノイズの分散 が、95%の確率で入る範囲
wn_band <- function(m) {
  df <- 2 * (2 * m + 1)
  qchisq(c(0.025, 0.975), df) / df
}

$m$ を1、8、32と変えて、理論のスペクトル密度（赤）と比べます。

In [ ]:
by_m <- do.call(rbind, lapply(c(1, 8, 32), function(m_i)
  data.frame(m = m_i, panel = sprintf("m = %d (%d frequencies)", m_i, 2 * m_i + 1),
             smooth_pgram(pg_ar2, m_i))))
by_m$panel <- factor(by_m$panel, levels = unique(by_m$panel))
ggplot(by_m, aes(freq)) +
  geom_line(aes(y = spec)) +
  geom_line(aes(y = truth), colour = "red") +
  facet_wrap(~panel) +
  labs(x = "Frequency (cycles per step)", y = "Smoothed periodogram (red: true)")
# 推定の誤差（理論のスペクトル密度とのRMSE）
by_m |>
  group_by(m) |>
  summarise(rmse = sqrt(mean((spec - truth)^2)))

- $m = 1$：まだ大きくばらつきます。
- $m = 8$：赤い線に近づきます。
- $m = 32$：なめらかですが、山が低く、谷が浅くなります。広い範囲を平均したので、山がならされました。

$m$ が小さいとばらつき、大きいと形がつぶれます。500本の系列でくり返して、誤差が最も小さくなる $m$ を探します。

In [ ]:
set.seed(1)
ms <- c(0, 1, 2, 4, 8, 16, 32, 64)
err <- replicate(500, {
  pg_i <- pgram(as.numeric(arima.sim(list(ar = ar2), n = n_ar)))
  pg_i$truth <- arma_spec(pg_i$freq, ar2)
  sapply(ms, function(m_i) {
    s <- smooth_pgram(pg_i, m_i)
    sqrt(mean((s$spec - s$truth)^2))
  })
})
rep_m <- data.frame(m = ms, frequencies = 2 * ms + 1, rmse = rowMeans(err))
ggplot(rep_m, aes(frequencies, rmse)) +
  geom_line() +
  geom_point() +
  scale_x_log10(breaks = rep_m$frequencies) +
  labs(x = "Number of frequencies averaged (2m + 1, log scale)", y = "RMSE (mean over 500 series)")
round(rep_m, 2)

誤差は、$m$ を大きくするといったん下がり、そのあと上がります。ちょうどよい幅が、途中にあります。

平滑化したピリオドグラムを本当のスペクトル密度で割った比は、どの範囲に入るでしょうか。
関数 `wn_band()` は、自由度 $2(2m+1)$ のカイ2乗分布から、比が95%の確率で入る範囲を出します。$m = 8$ で確かめます。

In [ ]:
s8 <- smooth_pgram(pg_ar2, 8)
band8 <- wn_band(8)
round(band8, 2)
mean(s8$spec / s8$truth > band8[1] & s8$spec / s8$truth < band8[2])     # 範囲に入った周波数の割合

この範囲は、B節で、残差がホワイトノイズかどうかを調べるのに使います。

### A-8. モデルを使って推定する

スペクトル密度を推定する方法は、もう1つあります。ARモデルを当てはめ、推定した係数をA-5の式に入れる方法です。

In [ ]:
fit_ar2 <- tsibble(t = 1:n_ar, x = x_ar2, index = t) |>
  model(ar = ARIMA(x ~ 0 + pdq(2, 0, 0)))
ar_est <- tidy(fit_ar2)$estimate               # 推定したARの係数
s2_est <- as.numeric(glance(fit_ar2)$sigma2)   # 推定したノイズの分散
round(c(phi1 = ar_est[1], phi2 = ar_est[2], sigma2 = s2_est), 3)
two <- rbind(
  data.frame(method = "smoothed periodogram (m = 8)", freq = s8$freq, spec = s8$spec),
  data.frame(method = "AR(2) model", freq = pg_ar2$freq, spec = arma_spec(pg_ar2$freq, ar_est, sigma2 = s2_est)))
ggplot(two, aes(freq, spec)) +
  geom_line(data = pg_ar2, aes(y = truth), colour = "red", linewidth = 1.5, alpha = 0.5) +
  geom_line(aes(colour = method)) +
  labs(x = "Frequency (cycles per step)", y = "Spectral density (red: true)", colour = "Estimate")
# 推定の誤差（理論のスペクトル密度とのRMSE）
two |>
  group_by(method) |>
  summarise(rmse = sqrt(mean((spec - arma_spec(freq, ar2))^2)))

- ARモデルによる推定は、なめらかで、赤い線にほぼ重なります。係数を2つ推定するだけだからです。
- ただし、これは「データがAR(2)から出てきた」と知っているからです。モデルが合わなければ、形ごと外れます。
- 平滑化したピリオドグラムは、モデルを仮定しません。その代わりに、ばらつきが残ります。

## B. 実データを解析する

Rに付属する`sunspot.year`を使います。太陽の黒点の数の、年ごとの値です。

B節は、A節で作った関数（`pgram`、`arma_spec`、`smooth_pgram`、`wn_band`）を使います。A節を実行してから進んでください。

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
year_from <- 1700       # 始まりの年：1700〜1800
year_to   <- 1988       # 終わりの年：1950〜1988
transform <- "none"     # 変換："none"（そのまま）か "sqrt"（平方根をとる）
m         <- 2          # 平滑化の幅：0〜6（となり合う 2m+1 個の周波数を平均する）
p         <- 2          # ARモデルの次数：1〜9
h         <- 22         # テストデータの年数：11 か 22

### B-1. データを確かめる

`sunspot.year`は、Rの古い形式（`ts`）の時系列です。`tsibble`に直してから確かめます。

In [ ]:
class(sunspot.year)
c(start = start(sunspot.year)[1], end = end(sunspot.year)[1], frequency = frequency(sunspot.year))
sun_all <- as_tsibble(sunspot.year) |>
  rename(year = index, number = value)
glimpse(sun_all)
interval(sun_all)
c(rows = nrow(sun_all), missing = sum(is.na(sun_all$number)), gaps = has_gaps(sun_all)$.gaps)

- 1行は1年です。`number`が、その年の黒点の数（黒点相対数の年平均）です。
- 期間は1700年から1988年、289行です。
- 欠測（`NA`）も、行の抜けもありません。

### B-2. 前処理

期間を絞り、設定で選んだ変換をして、`y`という名前にします。最後の`h`年をテストデータにします。

In [ ]:
sun <- sun_all |>
  filter(year >= year_from, year <= year_to) |>
  mutate(y = if (transform == "sqrt") sqrt(number) else number)
y_lab <- if (transform == "sqrt") "Square root of sunspot number" else "Sunspot number"
n <- nrow(sun)
train <- sun |> filter(year <= year_to - h)
c(n = n, train = nrow(train), test = n - nrow(train))

In [ ]:
ggplot(sun, aes(year, y)) +
  annotate("rect", xmin = year_to - h + 0.5, xmax = year_to + 0.5, ymin = -Inf, ymax = Inf,
           alpha = 0.15, fill = "red") +
  geom_line() +
  labs(x = "Year", y = y_lab)

赤い帯がテストデータです。山と谷が、ほぼ決まった間隔でくり返しています。ただし、山の高さは毎回違います。

### B-3. 時間の側から見る：ACF

In [ ]:
train |>
  ACF(y, lag_max = 40) |>
  autoplot() +
  labs(x = "Lag h (year)", y = "Sample ACF")

ACFは波を打っています。A-5のAR(2)と同じ形です。波の周期を、この図から正確に読むのは難しいので、周波数の側から見ます。

### B-4. 周波数の側から見る：ピリオドグラム

In [ ]:
pg <- pgram(train$y)
head(pg[order(-pg$spec), ], 5)                 # 高い順に5つ
period_hat <- pg$period[which.max(pg$spec)]    # いちばん高い山の周期（年）
period_hat
ggplot(pg, aes(freq, spec)) +
  geom_line() +
  geom_point(size = 0.8) +
  labs(x = "Frequency (cycles per year)", y = "Periodogram")

いちばん高い山の周期が、`period_hat`です。A-2の人工データと違い、山のまわりの周波数も高くなっています。低い周波数（長い周期）にも、小さな山があります。

### B-5. 平滑化する

山が高すぎて、ほかの部分が見えません。縦軸を対数の目盛りにして、平滑化したピリオドグラム（黒）を重ねます。

In [ ]:
pg_s <- smooth_pgram(pg, m)
ggplot(pg, aes(freq, spec)) +
  geom_line(colour = "grey60") +
  geom_line(data = pg_s, linewidth = 1) +
  scale_y_log10() +
  labs(x = "Frequency (cycles per year)", y = "Periodogram (black: smoothed, log scale)")

### B-6. ARモデルのスペクトル密度と比べる

設定セルの`p`で、AR(p)モデルを当てはめます。

In [ ]:
fit_ar <- train |>
  model(ar = ARIMA(y ~ 1 + pdq(p, 0, 0)))
coef_ar <- as.data.frame(tidy(fit_ar))[, c("term", "estimate", "std.error")]
coef_ar
ar_hat <- coef_ar$estimate[grepl("^ar", coef_ar$term)]    # ARの係数
sigma2_hat <- as.numeric(glance(fit_ar)$sigma2)           # ノイズの分散

推定した係数から、A-5の式でスペクトル密度を計算し（赤）、平滑化したピリオドグラム（黒）に重ねます。

In [ ]:
ar_curve <- data.frame(freq = pg$freq, spec = arma_spec(pg$freq, ar_hat, sigma2 = sigma2_hat))
fine <- seq(0.001, 0.5, 0.001)                             # 山の位置を探すための、細かい周波数
ar_period <- 1 / fine[which.max(arma_spec(fine, ar_hat))]  # ARモデルのスペクトル密度の、山の周期（年）
ggplot(pg, aes(freq, spec)) +
  geom_line(colour = "grey60") +
  geom_line(data = pg_s, linewidth = 1) +
  geom_line(data = ar_curve, colour = "red", linewidth = 1) +
  scale_y_log10() +
  labs(x = "Frequency (cycles per year)", y = "Spectrum (black: smoothed, red: AR)")
c(period_periodogram = period_hat, period_ar = ar_period)

赤い線が黒い線から離れている周波数では、モデルがデータの動きを写せていません。

### B-7. 診断：残差のスペクトルは平らか

モデルが従属性を写せていれば、残差はホワイトノイズに近くなります。ホワイトノイズのスペクトル密度は平らです。
残差のピリオドグラムを平滑化し、ノイズの分散の推定値で割ります。平らなら、1のまわりに並びます。
青い破線は、ホワイトノイズなら95%の確率で入る範囲（A-7の`wn_band()`）です。

In [ ]:
res <- augment(fit_ar)$.innov
pg_res <- smooth_pgram(pgram(res), m)
pg_res$ratio <- pg_res$spec / mean(pgram(res)$spec)
band <- wn_band(m)
ggplot(pg_res, aes(freq, ratio)) +
  geom_hline(yintercept = 1, colour = "grey60") +
  geom_hline(yintercept = band, colour = "blue", linetype = "dashed") +
  geom_line() +
  scale_y_log10() +
  labs(x = "Frequency (cycles per year)", y = "Residual spectrum / mean (log scale)")
resid_out <- mean(pg_res$ratio < band[1] | pg_res$ratio > band[2])    # 範囲の外に出た周波数の割合
resid_out

### B-8. 予測を比べる

訓練データで当てはめ、テストデータの`h`年を予測します。比べるのは次の5つです。

- `mean`：平均法、`naive`：ナイーブ法（第2回のベンチマーク）
- `snaive`：季節ナイーブ法。周期には、ピリオドグラムで見つけた`period_hat`を整数に丸めた値を使います
- `wave`：周期`period_hat`の波を当てはめて、先へ伸ばす（A-3）
- `ar`：AR(p)モデル

In [ ]:
lag_s <- round(period_hat)      # 季節ナイーブ法の周期（年）
fit <- train |>
  model(mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = SNAIVE(y ~ lag(lag_s)),
        wave   = TSLM(y ~ fourier(period = period_hat, K = 1)),
        ar     = ARIMA(y ~ 1 + pdq(p, 0, 0)))
fc <- fit |> forecast(h = h)
autoplot(fc, sun |> filter(year > year_to - 4 * h), level = NULL) +
  labs(x = "Year", y = y_lab, colour = "Method")

In [ ]:
acc <- accuracy(fc, sun) |> select(.model, RMSE, MAE)
acc

### B-9. 時系列交差検証

訓練データを`h`年ずつ伸ばしながら、同じ評価を4回くり返します。周期（`period_hat`）と次数（`p`）は、B-8と同じものを使います。

In [ ]:
acc_cv <- sun |>
  filter(year <= year_to - h) |>
  stretch_tsibble(.init = n - 4 * h, .step = h) |>
  model(mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = SNAIVE(y ~ lag(lag_s)),
        wave   = TSLM(y ~ fourier(period = period_hat, K = 1)),
        ar     = ARIMA(y ~ 1 + pdq(p, 0, 0))) |>
  forecast(h = h) |>
  accuracy(sun) |>
  select(.model, RMSE, MAE)
acc_cv

読み方：

- B-4：人工データの山は1点だけでした。黒点の山は、幅をもっています。黒点の周期が、毎回少しずつ違うからです。
- B-6：AR(2)のスペクトル密度（赤）は、山の位置は合いますが、山の幅が広すぎます。低い周波数の山も写せていません。
- B-7：範囲の外に出た周波数が5%よりずっと多ければ、残差に周期が残っています。
- B-8、B-9：人工データでは、波を当てはめた予測（`wave`）が最も良くなりました。黒点では、そうなりません。周期と高さが毎回少しずつ違うので、決まった波は、先へ行くほど山の位置がずれます。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `year_from` | 1700〜1800 |
| `year_to` | 1950〜1988 |
| `transform` | `"none"` `"sqrt"` |
| `m` | 0〜6 |
| `p` | 1〜9 |
| `h` | 11、22 |

読むときの観点：

1. `p`を2から3、6、9と上げると、ARモデルのスペクトル密度（赤）は、平滑化したピリオドグラム（黒）にどこから近づくか。残差の診断（B-7）と、予測の誤差はどう変わるか。
2. `p = 1`にすると、赤い線に山は作れるか。A-5の4つのモデルの図と合わせて読む。
3. `m`を0、2、6と変えると、黒い線の山の高さと幅はどう変わるか。B-7の青い破線の幅はどう変わるか。
4. `transform`を`"sqrt"`にすると、ピリオドグラムの山のまわりと、予測の順位は変わるか。
5. 期間（`year_from`、`year_to`）を変えると、いちばん高い山の周期（`period_hat`）は変わるか。季節ナイーブ法の周期（`lag_s`）はどうか。
6. `h`を11にすると、5つの予測の順位は変わるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（1700〜1988年、変換なし、m = 2、AR(2)、22年先）の結果と比べてください。